<a href="https://colab.research.google.com/github/joj-ojo/Desafios-tecnicos/blob/main/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

In [3]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@desafio-tecnico-513817.iam.gserviceaccount.com


In [4]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1_6iS5pfCj7L5ps8D8hO03V-9DrqgPWOenQOyf-G9SXM"
NOME_TABELA = "pacientes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Posto_Saude | pacientes | https://docs.google.com/spreadsheets/d/1_6iS5pfCj7L5ps8D8hO03V-9DrqgPWOenQOyf-G9SXM


In [9]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["paciente_id", "nome_ficticio", "data_nascimento", "sexo", "bairro", "tipo_convenio"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [20]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados
    return {
        "paciente_id": fake.cpf(), #cria um cpf fake acionando função da bilbioteca faker
        "nome_ficticio": fake.name(), #cria um nome fake acionando função da bilbioteca faker
        "data_nascimento": fake.date_this_year().isoformat(),   # datas como texto
        "sexo": random.choice(["F", "M", "Outro"]), #seleciona uma das opções listadas aleatoriamente
        "bairro": fake.bairro(), #cria um  bairro fake acionando função da bilbioteca faker
        "tipo_convenio": random.choice(["SUS", "Particular", "Convenio"]) seleciona uma das opções listadas aleatoriamente
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


In [19]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 21
Cabeçalho confere: True
[['paciente_id', 'nome_ficticio', 'data_nascimento', 'sexo', 'bairro', 'tipo_convenio'], ['1', 'Brenda Alves', '2026-08-17', 'N/B', 'Glória', 'SUS'], ['2', 'Davi Miguel Sousa', '2026-02-01', 'F', 'Vila Pinho', 'Particular']]


**PARE AQUI NA SESSÃO**